In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

from mlxtend import __version__ as mlxtend_version
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

from scipy.stats import pearsonr

DATASET_FOLDER = "dataset/"

In [2]:
artists = pd.read_csv(DATASET_FOLDER + 'artists.csv', sep=';')
tracks = pd.read_csv(DATASET_FOLDER + 'tracks.csv', sep=',')

artists_p = artists.add_prefix("artist_")
tracks_p = tracks.add_prefix("track_")

merged = tracks_p.merge(
    artists_p,
    how="left",
    left_on="track_id_artist",
    right_on="artist_id_author")

pair_cols = ["track_id", "artist_id_author"]
merged = merged.drop_duplicates(subset=pair_cols, keep="first")

In [3]:
bpm = pd.to_numeric(merged["track_bpm"], errors="coerce")
pop = pd.to_numeric(merged["track_popularity"], errors="coerce")
dur = pd.to_numeric(merged["track_duration_ms"], errors="coerce")
swear_it = pd.to_numeric(merged["track_swear_IT"], errors="coerce")
swear_en = pd.to_numeric(merged["track_swear_EN"], errors="coerce")
swear_total = swear_it.add(swear_en, fill_value=0)

duration_min = dur / 60000
swear_density = swear_total / duration_min
swear_density = swear_density.replace([np.inf, -np.inf], np.nan)

encoded = pd.get_dummies([])

encoded["bpm"] = pd.cut(
    bpm,
    bins=[-np.inf, 90, 120, np.inf],
    labels=["Slow", "Medium", "Fast"]
 )

encoded["popularity"] = pd.cut(
    pop,
    bins=[-np.inf, 3e1, np.inf],
    labels=["Low", "High"]
 )

encoded["duration"] = pd.cut(
    dur,
    bins=[-np.inf, 180000, 300000, np.inf],
    labels=["Short", "Medium", "Long"]
 )

median_swear_density = swear_density.median(skipna=True)
encoded["swear_density"] = pd.cut(
    swear_density,
    bins=[-np.inf, median_swear_density, np.inf],
    labels=["Low", "High"]
 )

encoded["lexical_density"] = pd.cut(
    pd.to_numeric(merged["track_lexical_density"], errors="coerce"),
    bins=[-np.inf, 0.5, np.inf],
    labels=["Low", "High"]
 )

 
encoded["loudness"] = pd.cut(
    pd.to_numeric(merged["track_loudness"], errors="coerce"),
    bins=[-np.inf, 27, 54, np.inf],
    labels=["Soft", "Medium", "Loud"]
 )

encoded["acuteness"] = pd.cut(
    pd.to_numeric(merged["track_centroid"], errors="coerce"),
    bins=[-np.inf, 0.15, np.inf],
    labels=["Low", "High"]
 )

encoded.head()

,bpm,popularity,duration,swear_density,lexical_density,loudness,acuteness
0,Fast,High,Medium,High,High,Soft,High
1,Fast,High,Medium,High,High,Soft,High
2,Fast,High,Medium,High,High,Medium,High
3,Fast,High,Short,High,High,Soft,Low
4,Medium,High,Medium,Low,Low,Soft,Low


In [4]:
transactions = encoded.apply(lambda row: [f"{col}_{row[col]}" for col in encoded.columns], axis=1).tolist()
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_array, columns=te.columns_)

df.head()



,acuteness_High,acuteness_Low,acuteness_nan,bpm_Fast,bpm_Medium,bpm_Slow,bpm_nan,duration_Long,duration_Medium,duration_Short,...,loudness_Loud,loudness_Medium,loudness_Soft,loudness_nan,popularity_High,popularity_Low,popularity_nan,swear_density_High,swear_density_Low,swear_density_nan
0,True,False,False,True,False,False,False,False,True,False,...,False,False,True,False,True,False,False,True,False,False
1,True,False,False,True,False,False,False,False,True,False,...,False,False,True,False,True,False,False,True,False,False
2,True,False,False,True,False,False,False,False,True,False,...,False,True,False,False,True,False,False,True,False,False
3,False,True,False,True,False,False,False,False,False,True,...,False,False,True,False,True,False,False,True,False,False
4,False,True,False,False,True,False,False,False,True,False,...,False,False,True,False,True,False,False,False,True,False


In [5]:
frequent_itemsets = apriori(df, min_support=2e-1, use_colnames=True)
frequent_itemsets.sort_values("support", ascending=False).head(20)

,support,itemsets
1,0.683508,(acuteness_Low)
5,0.620711,(duration_Medium)
10,0.610141,(loudness_Soft)
7,0.598764,(lexical_density_High)
11,0.521813,(popularity_High)
14,0.496551,(swear_density_Low)
13,0.496462,(swear_density_High)
12,0.475589,(popularity_Low)
22,0.429992,"(acuteness_Low, loudness_Soft)"
17,0.420407,"(acuteness_Low, duration_Medium)"


In [6]:
rules_conf = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.6,
    num_itemsets=len(df.index),
)

columns_to_show = [
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift",
    "leverage",
]

rules_conf[columns_to_show].sort_values(["lift", "leverage"], ascending=False).head(30)

,antecedents,consequents,support,confidence,lift,leverage
47,"(acuteness_Low, popularity_Low)",(loudness_Soft),0.214727,0.682906,1.119260,0.022880
29,(swear_density_High),(lexical_density_High),0.331273,0.667268,1.114410,0.034010
44,"(acuteness_Low, swear_density_High)",(lexical_density_High),0.209263,0.656549,1.096507,0.018418
49,"(acuteness_Low, swear_density_Low)",(loudness_Soft),0.239900,0.667498,1.094006,0.020614
26,(duration_Short),(lexical_density_High),0.217773,0.653319,1.091113,0.018185
50,"(loudness_Soft, swear_density_Low)",(acuteness_Low),0.239900,0.740597,1.083524,0.018493
54,"(swear_density_Low, duration_Medium)",(loudness_Soft),0.208457,0.658834,1.079806,0.015407
27,(loudness_Medium),(lexical_density_High),0.245275,0.640318,1.069400,0.015917
32,(swear_density_Low),(loudness_Soft),0.323927,0.652354,1.069187,0.020961
18,(lexical_density_Low),(duration_Medium),0.261489,0.662957,1.068060,0.016663


In [ ]:
r'''
df["swear_high_short"] = df["swear_density_High"] & df["duration_Short"]
df["swear_low_soft"] = df["swear_density_Low"] & df["loudness_Soft"]
df["swear_low_popular"] = df["swear_density_Low"] & df["popularity_High"]
df["lexical_low_soft"] = df["lexical_density_Low"] & df["loudness_Soft"]
'''

#NEW:

df["obtuse_unpopular"] = df["acuteness_Low"] & df["popularity_Low"]
df["polite_medium"] = df["swear_density_Low"] & df["duration_Medium"]
df["soft_popular"] = df["loudness_Soft"] & df["popularity_High"]
df["polite_soft"] = df["swear_density_Low"] & df["loudness_Soft"]